![](http://labs.criteo.com/wp-content/uploads/2017/08/CustomersWhoBought3.jpg)

En este cuaderno, se busca  implementar algunos algoritmos de recomendación (basados en contenido, en popularidad y en filtrado colaborativo) y se va a construir un conjunto (ensemble) de estos modelos para crear nuestro sistema de recomendación final. Contamos con dos conjuntos de datos de MovieLens.

El conjunto de datos completo: Consiste en 26,000,000 de calificaciones y 750,000 aplicaciones de etiquetas aplicadas a 45,000 películas por 270,000 usuarios. Incluye datos del genoma de etiquetas con 12 millones de puntuaciones de relevancia en 1,100 etiquetas.

El conjunto de datos pequeño: Comprende 100,000 calificaciones y 1,300 aplicaciones de etiquetas aplicadas a 9,000 películas por 700 usuarios.

Construye un **Recomendador Simple** utilizando películas del conjunto de datos completo, mientras que todos los sistemas de recomendación personalizados harán uso del conjunto de datos pequeño (debido a que mi capacidad de cómputo es muy limitada). Como primer paso, construiré mi sistema de recomendación simple.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

import warnings
warnings.simplefilter('ignore')




### Análisis de datos y manipulación
import pandas as pd  # Estructuras de datos flexibles y herramientas de análisis (DataFrames, Series)

import numpy as np   # Computación numérica con arrays multidimensionales y funciones matemáticas

### Visualización de datos
import matplotlib.pyplot as plt  # Biblioteca fundamental para crear visualizaciones estáticas, animadas e interactivas

import seaborn as sns            # Biblioteca de visualización basada en matplotlib con estilos atractivos y gráficos estadísticos

### Análisis estadístico
from scipy import stats  # Funciones estadísticas (distribuciones, tests estadísticos, medidas de correlación)

### Evaluación segura de strings
from ast import literal_eval  # Convierte strings que contienen estructuras de Python en las estructuras mismas

### Procesamiento de texto y machine learning
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer  # Conversión de texto a vectores numéricos

from sklearn.metrics.pairwise import linear_kernel, cosine_similarity  # Cálculo de similitudes entre vectores

### Procesamiento de lenguaje natural (NLP)
from nltk.stem.snowball import SnowballStemmer  # Algoritmo de stemming para reducir palabras a su raíz

from nltk.stem.wordnet import WordNetLemmatizer # Lematización para reducir palabras a su forma canónica

from nltk.corpus import wordnet                 # Base de datos léxica del inglés para lematización

### Sistema de recomendación
from surprise import Reader, Dataset, SVD       # Biblioteca para sistemas de recomendación (Reader: formato de datos, Dataset: conjunto de datos, SVD: Descomposición en Valores Singulares)

from surprise.model_selection import cross_validate  # Validación cruzada para modelos de recomendación


## Recomendador Simple 

El Recomendador Simple ofrece recomendaciones generales a todos los usuarios basadas en la popularidad de las películas y (a veces) en su género. La idea básica detrás de este recomendador es que las películas más populares y con mejores críticas tienen una mayor probabilidad de ser del agrado del público promedio. Este modelo no proporciona recomendaciones personalizadas basadas en el usuario (como nuestro reto).

La implementación de este modelo es extremadamente sencilla. Todo lo que debemos hacer es ordenar las películas según sus calificaciones y popularidad, y mostrar las películas principales de la lista. Como paso adicional, podemos incluir un argumento de género para obtener las mejores películas de un género en particular.

Cargar los datos

In [ ]:
md = pd.read_csv('.../movies_metadata.csv')
md.head()

que hace la linea siguiente

In [ ]:
md['genres'] = md['genres'].fillna('[]').apply(literal_eval).apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

Usaremos las calificaciones de TMDb para crear nuestro Ranking de las Mejores Películas (Top Movies Chart). La fórmula de calificación ponderada de IMDB para construir este ranking. Matemáticamente, se representa de la siguiente manera:

Calificación Ponderada (WR) = $(\frac{v}{v + m} \cdot R) + (\frac{m}{v + m} \cdot C)$

donde,

v es el número de votos que tiene la película

m es el número mínimo de votos requeridos para ser incluida en el ranking

R es la calificación promedio de la película

C es la calificación promedio de todas las películas del conjunto de datos

El siguiente paso es determinar un valor apropiado para m, el número mínimo de votos necesarios para aparecer en el ranking. Usaremos el percentil 95 como punto de corte. En otras palabras, para que una película aparezca en el ranking, debe tener más votos que al menos el 95% de las películas en la lista.

El ranking general de las 250 mejores películas y definiré una función para crear rankings por género específico. 

In [ ]:
vote_counts = md[md['vote_count'].notnull()]['vote_count'].astype('int')
vote_averages = md[md['vote_average'].notnull()]['vote_average'].astype('int')
C = vote_averages.mean()
C

In [ ]:
m = vote_counts.quantile(0.95)
m

In [ ]:
md['year'] = pd.to_datetime(md['release_date'], errors='coerce').apply(lambda x: str(x).split('-')[0] if x != np.nan else np.nan)

In [ ]:
qualified = md[(md['vote_count'] >= m) & (md['vote_count'].notnull()) & (md['vote_average'].notnull())][['title', 'year', 'vote_count', 'vote_average', 'popularity', 'genres']]
qualified['vote_count'] = qualified['vote_count'].astype('int')
qualified['vote_average'] = qualified['vote_average'].astype('int')
qualified.shape

Por lo tanto, para calificar y ser considerada en el ranking, una película debe tener al menos 434 votos en TMDB. También observamos que la calificación promedio de una película en TMDB es de 5.244 en una escala de 10. En total, 2274 películas califican para estar en nuestro ranking.

In [ ]:
def weighted_rating(x):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

In [ ]:
qualified['wr'] = qualified.apply(weighted_rating, axis=1)

In [ ]:
qualified = qualified.sort_values('wr', ascending=False).head(250)

### Películas Top 

In [ ]:
qualified.head(15)

Vemos que tres películas de Christopher Nolan —Inception, The Dark Knight e Interstellar, aparecen en los primeros lugares de nuestro ranking. El gráfico también muestra una fuerte preferencia de los usuarios de TMDB hacia ciertos géneros y directores.

Ahora construiremos una función que genere rankings para géneros específicos. Para ello, relajaremos nuestras condiciones por defecto utilizando el percentil 85 en lugar del 95.

In [ ]:
s = md.apply(lambda x: pd.Series(x['genres']),axis=1).stack().reset_index(level=1, drop=True)
s.name = 'genre'
gen_md = md.drop('genres', axis=1).join(s)

In [ ]:
def build_chart(genre, percentile=0.85):
    df = gen_md[gen_md['genre'] == genre]
    vote_counts = df[df['vote_count'].notnull()]['vote_count'].astype('int')
    vote_averages = df[df['vote_average'].notnull()]['vote_average'].astype('int')
    C = vote_averages.mean()
    m = vote_counts.quantile(percentile)
    
    qualified = df[(df['vote_count'] >= m) & (df['vote_count'].notnull()) & (df['vote_average'].notnull())][['title', 'year', 'vote_count', 'vote_average', 'popularity']]
    qualified['vote_count'] = qualified['vote_count'].astype('int')
    qualified['vote_average'] = qualified['vote_average'].astype('int')
    
    qualified['wr'] = qualified.apply(lambda x: (x['vote_count']/(x['vote_count']+m) * x['vote_average']) + (m/(m+x['vote_count']) * C), axis=1)
    qualified = qualified.sort_values('wr', ascending=False).head(250)
    
    return qualified

Veamos el método mostrando el Top 15 de películas de romance (el género de romance casi no apareció en nuestro ranking general, a pesar de ser uno de los géneros cinematográficos más populares).

### Top 15 de películas de romance

In [ ]:
build_chart('Romance').head(15)

La mejor película de romance según nuestras métricas es la producción de Bollywood Dilwale Dulhania Le Jayenge. Esta cinta protagonizada por Shahrukh Khan.

## Recomendador Basado en Contenido

El recomendador que construimos en la sección anterior presenta algunas limitaciones importantes. Por ejemplo, ofrece las mismas recomendaciones a todos los usuarios, sin tener en cuenta sus gustos personales. Si una persona que ama las películas románticas (y detesta la acción) revisará nuestro Top 15, probablemente no disfrutaría de la mayoría de las películas. Incluso si revisara nuestros rankings por género, aún no estaría recibiendo las mejores recomendaciones.

Por ejemplo, consideremos a una persona que ama *Dilwale Dulhania Le Jayenge*, *My Name is Khan* y *Kabhi Khushi Kabhi Gham*. Podemos inferir que esta persona disfruta de las películas protagonizadas por el actor **Shahrukh Khan** y dirigidas por **Karan Johar**. Aun si consultara el ranking de películas románticas, probablemente no encontraría estas entre las principales recomendaciones.

Para personalizar mejor nuestras recomendaciones, hay que construir un motor que calcule la **similitud entre películas** basándose en ciertos atributos, y que sugiera aquellas más parecidas a una película que le haya gustado al usuario. Dado que utilizaremos metadatos de las películas (o su contenido) para construir este motor, este enfoque se conoce como **Filtrado Basado en Contenido**.

Vamops a construir dos recomendadores basados en contenido, a partir de:
* Sinopsis y eslóganes de las películas  
* Reparto, equipo técnico, palabras clave y género  

Como se mencionó en la introducción, se utiliza un subconjunto de todas las películas disponibles debido a las limitaciones de capacidad de cómputo.


In [ ]:
links_small = pd.read_csv('.../links_small.csv')
links_small = links_small[links_small['tmdbId'].notnull()]['tmdbId'].astype('int')

In [ ]:
md = md.drop([19730, 29503, 35587])

In [ ]:
#Check EDA Notebook for how and why I got these indices.
md['id'] = md['id'].astype('int')

In [ ]:
smd = md[md['id'].isin(links_small)]
smd.shape

 Tenemos **9099** películas disponibles en nuestro conjunto de datos reducido de metadatos de películas,
 el cual es 5 veces más pequeño que nuestro conjunto de datos original de 45,000 películas.


### Recomendador Basado en la Descripción de la Película

Primero intentemos construir un recomendador utilizando las **descripciones** y **eslóganes** de las películas.  
No contamos con una métrica cuantitativa para evaluar el rendimiento de nuestro modelo, por lo que esta evaluación deberá realizarse de forma **cualitativa**.


In [ ]:
smd['tagline'] = smd['tagline'].fillna('')
smd['description'] = smd['overview'] + smd['tagline']
smd['description'] = smd['description'].fillna('')

In [ ]:
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')
tfidf_matrix = tf.fit_transform(smd['description'])

In [ ]:
tfidf_matrix.shape

#### Similitud del Coseno

Vamos a utilizar la **Similitud del Coseno** para calcular una cantidad numérica que indique el grado de similitud entre dos películas.  
Matemáticamente, se define de la siguiente manera:

$cosine(x,y) = \frac{x. y^\intercal}{||x||.||y||} $

Dado que hemos utilizado el **Vectorizador TF-IDF**, calcular el **producto punto** nos dará directamente el valor de la similitud del coseno.  
Por lo tanto, utilizaremos la función **linear_kernel** de *sklearn* en lugar de *cosine_similarities*, ya que es mucho más rápida.


In [ ]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [ ]:
cosine_sim[0]

Ahora tenemos una **matriz de similitud del coseno por pares** para todas las películas de nuestro conjunto de datos.  
El siguiente paso es escribir una función que devuelva las **30 películas más similares** basándose en el puntaje de similitud del coseno.


In [ ]:
smd = smd.reset_index()
titles = smd['title']
indices = pd.Series(smd.index, index=smd['title'])

In [ ]:
def get_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:31]
    movie_indices = [i[0] for i in sim_scores]
    return titles.iloc[movie_indices]

Ahora intentemos obtener las **principales recomendaciones** para algunas películas y ver qué tan buenas son las sugerencias.


In [ ]:
get_recommendations('The Godfather').head(10)

In [ ]:
get_recommendations('The Dark Knight').head(10)

Podemos observar que para **The Dark Knight**, nuestro sistema logra identificarla como una película de *Batman* y, en consecuencia, recomienda otras películas de *Batman* como sus principales sugerencias.  
Sin embargo, lamentablemente eso es todo lo que este sistema puede hacer por el momento.  
Esto no resulta muy útil para la mayoría de las personas, ya que no toma en cuenta características muy importantes como el **elenco**, **equipo técnico**, **director** y **género**, los cuales influyen en la calificación y popularidad de una película.  
Alguien a quien le gustó **The Dark Knight** probablemente la disfruta más por **Nolan**, y seguramente detestaría **Batman Forever** y otras películas de menor calidad dentro de la franquicia de Batman.

Por lo tanto, pueden estudiar lo que se haría con **metadatos** mucho más representativos que solo la **sinopsis** y el **eslogan**.  
En la siguiente subsección se construye un recomendador más sofisticado que tenga en cuenta el **género**, las **palabras clave**, el **reparto** y el **equipo técnico**.


### Recomendador Basado en Metadatos

Para construir nuestro recomendador estándar basado en contenido y metadatos, necesitaremos combinar nuestro conjunto de datos actual con los conjuntos de datos de **equipo técnico (crew)** y **palabras clave (keywords)**.  
Prepararemos estos datos como **primer paso**.


In [ ]:
credits = pd.read_csv('../credits.csv')
keywords = pd.read_csv('../keywords.csv')

In [ ]:
keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
md['id'] = md['id'].astype('int')

In [ ]:
md.shape

In [ ]:
md = md.merge(credits, on='id')
md = md.merge(keywords, on='id')

In [ ]:
smd = md[md['id'].isin(links_small)]
smd.shape

Ahora tenemos el **reparto**, el **equipo técnico**, los **géneros** y los **créditos**, todos en un mismo *dataframe*.  
Vamos a procesar estos datos un poco más utilizando las siguientes intuiciones:

1. **Equipo técnico (Crew):**  
   Del equipo técnico, solo tomaremos al **director** como característica, ya que los demás miembros no influyen tanto en la *esencia* de la película.

2. **Reparto (Cast):**  
   La selección del reparto es un poco más complicada.  
   Los actores poco conocidos o los papeles menores no afectan demasiado la opinión del público sobre una película.  
   Por lo tanto, debemos seleccionar solo los **personajes principales** y sus respectivos actores.  
   De manera arbitraria, elegiremos a los **3 actores principales** que aparecen en la lista de créditos.


In [ ]:
smd['cast'] = smd['cast'].apply(literal_eval)
smd['crew'] = smd['crew'].apply(literal_eval)
smd['keywords'] = smd['keywords'].apply(literal_eval)
smd['cast_size'] = smd['cast'].apply(lambda x: len(x))
smd['crew_size'] = smd['crew'].apply(lambda x: len(x))

In [ ]:
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [ ]:
smd['director'] = smd['crew'].apply(get_director)

In [ ]:
smd['cast'] = smd['cast'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])
smd['cast'] = smd['cast'].apply(lambda x: x[:3] if len(x) >=3 else x)

In [ ]:
smd['keywords'] = smd['keywords'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

El enfoque para construir este recomendador va a ser *experimental*.  
Se planea crear un **volcado de metadatos** (*metadata dump*) para cada película, que contenga los siguientes elementos:  
**géneros, director, actores principales y palabras clave.**  
Luego, se utiliza un **Count Vectorizer** para generar nuestra matriz de conteo, tal como hicimos en el recomendador basado en descripciones.  
Los pasos restantes serán similares a los anteriores: calcularemos las **similitudes del coseno** y devolveremos las películas más parecidas.

Estos son los pasos que sigo para preparar los datos de géneros y créditos:

1. **Eliminar espacios y convertir todo a minúsculas** en todas las características.  
   De esta manera, nuestro motor no confundirá nombres como **Johnny Depp** y **Johnny Galecki**.  

2. **Mencionar al director tres veces** para darle un mayor peso en comparación con el resto del reparto.


In [ ]:
smd['cast'] = smd['cast'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

In [ ]:
smd['director'] = smd['director'].astype('str').apply(lambda x: str.lower(x.replace(" ", "")))
smd['director'] = smd['director'].apply(lambda x: [x,x, x])

#### Palabras Clave (Keywords)

Realizaremos una pequeña **preparación previa** de nuestras palabras clave antes de utilizarlas.  
Como primer paso, calcularemos la **frecuencia de aparición** de cada palabra clave que aparece en el conjunto de datos.


In [ ]:
s = smd.apply(lambda x: pd.Series(x['keywords']),axis=1).stack().reset_index(level=1, drop=True)
s.name = 'keyword'

In [ ]:
s = s.value_counts()
s[:5]

Las palabras clave aparecen con frecuencias que van de **1 a 610**.  
No nos sirven aquellas palabras clave que aparecen **solo una vez**, por lo que podemos eliminarlas sin problema.  
Finalmente, convertiremos cada palabra a su **raíz léxica (stem)**, de modo que términos como *Dogs* y *Dog* se consideren equivalentes.


In [ ]:
s = s[s > 1]

In [ ]:
stemmer = SnowballStemmer('english')
stemmer.stem('dogs')

In [ ]:
def filter_keywords(x):
    words = []
    for i in x:
        if i in s:
            words.append(i)
    return words

In [ ]:
smd['keywords'] = smd['keywords'].apply(filter_keywords)
smd['keywords'] = smd['keywords'].apply(lambda x: [stemmer.stem(i) for i in x])
smd['keywords'] = smd['keywords'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

In [ ]:
smd['soup'] = smd['keywords'] + smd['cast'] + smd['director'] + smd['genres']
smd['soup'] = smd['soup'].apply(lambda x: ' '.join(x))

In [ ]:
count = CountVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')
count_matrix = count.fit_transform(smd['soup'])

In [ ]:
cosine_sim = cosine_similarity(count_matrix, count_matrix)

In [ ]:
smd = smd.reset_index()
titles = smd['title']
indices = pd.Series(smd.index, index=smd['title'])

Reutilizaremos la función **get_recommendations** que escribimos anteriormente.  
Dado que nuestras puntuaciones de **similitud del coseno** han cambiado, esperamos obtener resultados diferentes (y probablemente mejores).  
Verifiquemos nuevamente el caso de **The Dark Knight** y veamos qué recomendaciones obtenemos esta vez.


In [ ]:
get_recommendations('The Dark Knight').head(10)

Podemos estar mucho más satisfechos con los resultados obtenidos esta vez.  
Las recomendaciones parecen haber identificado correctamente otras películas de **Christopher Nolan** (debido al alto peso asignado al director) y las han colocado entre las principales sugerencias.  
Disfruté ver **The Dark Knight**, así como otras películas de la lista, incluyendo **Batman Begins**, **The Prestige** y **The Dark Knight Rises**.

Por supuesto, podemos seguir experimentando con este motor probando diferentes **pesos para nuestras características** (directores, actores, géneros),  
limitando la cantidad de **palabras clave** utilizadas en la mezcla (*soup*),  
ponderando los **géneros según su frecuencia**,  
mostrando solo películas del **mismo idioma**, entre otras posibles mejoras.


Veamos también las recomendaciones para otra película: **Mean Girls**,  
que resulta ser la película favorita de mi novia.



In [ ]:
get_recommendations('Mean Girls').head(10)

#### Popularidad y Calificaciones

Una cosa que notamos acerca de nuestro sistema de recomendación es que sugiere películas **sin tener en cuenta las calificaciones ni la popularidad**.  
Es cierto que **Batman and Robin** comparte muchos personajes con **The Dark Knight**, pero fue una película terrible que no debería recomendarse a nadie.

Por lo tanto, añadiremos un mecanismo para **filtrar las malas películas** y devolver únicamente aquellas que sean **populares** y que hayan tenido una **buena recepción crítica**.

Tomaremos las **25 películas principales** basadas en los puntajes de similitud y calcularemos el número de votos correspondiente al **percentil 60**.  
Luego, usando este valor como \( m \), calcularemos la **calificación ponderada** de cada película utilizando la **fórmula de IMDB**, tal como hicimos en la sección del *Recomendador Simple*.


In [ ]:
def improved_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:26]
    movie_indices = [i[0] for i in sim_scores]
    
    movies = smd.iloc[movie_indices][['title', 'vote_count', 'vote_average', 'year']]
    vote_counts = movies[movies['vote_count'].notnull()]['vote_count'].astype('int')
    vote_averages = movies[movies['vote_average'].notnull()]['vote_average'].astype('int')
    C = vote_averages.mean()
    m = vote_counts.quantile(0.60)
    qualified = movies[(movies['vote_count'] >= m) & (movies['vote_count'].notnull()) & (movies['vote_average'].notnull())]
    qualified['vote_count'] = qualified['vote_count'].astype('int')
    qualified['vote_average'] = qualified['vote_average'].astype('int')
    qualified['wr'] = qualified.apply(weighted_rating, axis=1)
    qualified = qualified.sort_values('wr', ascending=False).head(10)
    return qualified

In [ ]:
improved_recommendations('The Dark Knight')

Veamos también las recomendaciones para **Mean Girls**.


In [ ]:
improved_recommendations('Mean Girls')

Desafortunadamente, **Batman and Robin** no desaparece de nuestra lista de recomendaciones.  
Esto probablemente se deba a que tiene una calificación de **4**, que está solo ligeramente por debajo del promedio en TMDB.  
Ciertamente no merece un 4 cuando películas increíbles como **The Dark Knight Rises** apenas alcanzan un **7** (cuestión de gustos claro, jejeje).  
Sin embargo, no hay mucho que podamos hacer al respecto.

Por lo tanto, concluiremos aquí nuestra sección del **Recomendador Basado en Contenido**.


## Filtrado Colaborativo

El recomendador basado en contenido muestra algunas limitaciones importantes. Solo es capaz de sugerir películas que son *similares* a una película en concreto. Es decir, no es capaz de capturar gustos y proporcionar recomendaciones a través de diferentes géneros.

Además, el recomendador que construimos no es realmente personal, ya que no captura los gustos personales y los sesgos de un usuario. Cualquier persona que lo consulte para obtener recomendaciones basadas en una película recibirá las mismas recomendaciones para esa película, sin importar quién sea.

Por lo tanto, en esta sección, usaremos una técnica llamada **Filtrado Colaborativo** para hacer recomendaciones a los espectadores. El Filtrado Colaborativo se basa en la idea de que usuarios similares a uno pueden usarse para predecir cuánto me gustará un producto o servicio particular que esos usuarios han usado/experimentado pero yo no.

No vamos a implementar el Filtrado Colaborativo desde cero. En su lugar, usaremos la librería **Surprise**, que utiliza algoritmos extremadamente potentes como la **Descomposición en Valores Singulares (SVD)** para minimizar el RMSE (Error Cuadrático Medio) y dar mejores recomendaciones.

## Descomposición en Valores Singulares (SVD)

La **Descomposición en Valores Singulares (SVD)** es una técnica fundamental del álgebra lineal que descompone una matriz en tres matrices componentes. Para el contexto de sistemas de recomendación, se aplica a la matriz usuario-ítem para descubrir patrones latentes en los datos.

### Representación Matemática

Dada una matriz A de dimensiones m×n, la SVD la factoriza como:

**A = U × Σ × Vᵀ**

Donde:
- **U**: Matriz ortogonal m×m (vectores singulares izquierdos)
- **Σ**: Matriz diagonal m×n (valores singulares en orden descendente)
- **Vᵀ**: Matriz ortogonal n×n transpuesta (vectores singulares derechos)

### Aplicación en Sistemas de Recomendación

En filtrado colaborativo, la SVD se utiliza para:

- **Reducir dimensionalidad**: Mantener solo los k valores singulares más importantes
- **Capturar factores latentes**: Los valores singulares representan "factores" o "características ocultas" que explican las preferencias de los usuarios
- **Predecir calificaciones**: Completar los valores faltantes en la matriz usuario-ítem

### Ventajas en Surprise

La librería Surprise optimiza SVD para:
- Manejar datos dispersos eficientemente
- Minimizar el error de predicción (RMSE)
- Escalar a grandes conjuntos de datos
- Proporcionar recomendaciones personalizadas y precisas

In [ ]:
#Tomados el rating de 1 a 5 estrellas
reader = Reader(rating_scale=(1, 5))

In [ ]:
#
ratings = pd.read_csv('../ratings_small.csv')
ratings.head()

In [ ]:
#Cargamos las calificaciones de los usuarios a las pelis
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

In [ ]:
svd = SVD()
results = cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=5, verbose=True) # cv=5, usa 5-fold cross validation (divide datos en 5 partes, usa 4 para entrenar y 1 para test)


Obtenemos un **Error Cuadrático Medio** promedio de 0.8970, lo cual es más que suficiente para nuestro caso. Procedamos ahora a entrenar con nuestro dataset completo y generar las predicciones.

In [ ]:
trainset = data.build_full_trainset() #Se usa ntodos los datos
svd.fit(trainset)

*Actividad 1*

Selecciona al usuario 1 y revisa las calificaciones que ha dado.

In [ ]:
# Completa. Usa ratings[]

In [ ]:
# Genera una Predicción para el usuario 1 y ve si puso 3 de calificación en la película 302
# usa svd.predict ()


Por ejemplo puedes obtener esto para la película 302

Prediction(uid=1, iid=302, r_ui=3, est=2.5027733276596384, details={'was_impossible': False})

Para la película con ID 302, obtenemos una predicción estimada de **2.502**. Una característica sorprendente de este sistema de recomendación es que no le importa qué sea la película (o qué contenido tenga). Funciona puramente en base a un ID de película asignado e intenta predecir las calificaciones basándose en cómo los otros usuarios han calificado la película.

## Recomendador híbrido

![](https://www.toonpool.com/user/250/files/hybrid_20095.jpg)

En esta sección, se va a construir un recomendador híbrido simple que reúna las técnicas que hemos implementado en los motores basados en contenido y en filtrado colaborativo. Así es como funcionará:

* **Entrada:** ID de Usuario y el Título de una Película
* **Salida:** Películas similares ordenadas en base a las calificaciones esperadas por ese usuario en particular.

In [ ]:
def convert_int(x):
    try:
        return int(x)
    except:
        return np.nan

In [ ]:
id_map = pd.read_csv('../links_small.csv')[['movieId', 'tmdbId']]
id_map['tmdbId'] = id_map['tmdbId'].apply(convert_int)
id_map.columns = ['movieId', 'id']
id_map = id_map.merge(smd[['title', 'id']], on='id').set_index('title')

In [ ]:
indices_map = id_map.set_index('id')

In [ ]:
def hybrid(userId, title):
    idx = indices[title]
    tmdbId = id_map.loc[title]['id']
    #print(idx)
    movie_id = id_map.loc[title]['movieId']
    
    sim_scores = list(enumerate(cosine_sim[int(idx)]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:26]
    movie_indices = [i[0] for i in sim_scores]
    
    movies = smd.iloc[movie_indices][['title', 'vote_count', 'vote_average', 'year', 'id']]
    movies['est'] = movies['id'].apply(lambda x: svd.predict(userId, indices_map.loc[x]['movieId']).est)
    movies = movies.sort_values('est', ascending=False)
    return movies.head(10)

Prueba Avatar en dos usuarios, el 1 y alguno más

In [ ]:
#

In [ ]:
#

Podemos observar que para nuestro recomendador híbrido, obtenemos recomendaciones diferentes para distintos usuarios aunque la película de entrada sea la misma. Por lo tanto, nuestras recomendaciones son más personalizadas y adaptadas a usuarios particulares.

## Conclusión

En este notebook, hemos construido 4 sistemas de recomendación diferentes basados en distintas ideas y algoritmos. Estos son los siguientes:

1. **Recomendador Simple:** Este sistema utilizó el Conteo de Votos y los Promedios de Calificación generales de TMDB para crear Listas de las Mejores Películas, tanto en general como para un género específico. Se utilizó el Sistema de Ponderación de Calificaciones de IMDB para calcular las puntuaciones sobre las cuales finalmente se realizó la ordenación.

2. **Recomendador Basado en Contenido:** Construimos dos motores basados en contenido; uno que tomó como entrada la sinopsis y las taglines de las películas, y otro que utilizó metadatos como elenco, equipo, género y palabras clave para generar predicciones. También implementamos un filtro simple para dar mayor preferencia a las películas con más votos y mejores calificaciones.

3. **Filtrado Colaborativo:** Utilizamos la potente librería Surprise para construir un filtro colaborativo basado en descomposición en valores singulares. El RMSE obtenido fue menor que 1 y el motor proporcionó calificaciones estimadas para un usuario y película dados.

4. **Motor Híbrido:** Combinamos ideas del filtrado basado en contenido y colaborativo para construir un motor que proporcionó sugerencias de películas a un usuario particular basándose en las calificaciones estimadas que había calculado internamente para ese usuario.